In [2]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [3]:
import os

os.chdir(r'C:\Users\HP\OneDrive\Desktop\ANN')

In [4]:
#load ann trianed model
model = load_model(r'C:\Users\HP\OneDrive\Desktop\ANN\model.h5')
#load encoders and scaler
with open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender=pickle.load(file)
with open('one_hot_encoder_geo.pkl','rb') as file:
    one_hot_encoder_geo=pickle.load(file)
with open('scaler.pkl','rb') as file:
    scaler=pickle.load(file)


In [5]:
input_data={
    'CreditScore':600,
    'Geography':'France',
      'Gender':'Male',
      'Age':40,
      'Tenure':3,
      'Balance':60000,
      'NumOfProducts':2,
      'HasCrCard':1,
      'IsActiveMember':1,
      'EstimatedSalary':50000

}

In [6]:
def predict_customer(input_data):

    input_df = pd.DataFrame([input_data])

    input_df['Gender'] = label_encoder_gender.transform(
        input_df['Gender']
    )

    geo_encoded = one_hot_encoder_geo.transform(
        input_df[['Geography']]
    )

    geo_encoded_df = pd.DataFrame(
        geo_encoded,
        columns=one_hot_encoder_geo.get_feature_names_out(['Geography'])
    )

    input_df = input_df.drop('Geography', axis=1)

    input_df = pd.concat([
        input_df.drop('Gender', axis=1),
        geo_encoded_df,
        input_df[['Gender']]
    ], axis=1)

    # Match training column order
    input_df = input_df[scaler.feature_names_in_]

    # Scale
    input_scaled = scaler.transform(input_df)

    # Prediction
    probability = model.predict(input_scaled)[0][0]

    if probability >= 0.5:
        result = "Customer will exit"
    else:
        result = "Customer will stay"

    return probability, result

In [7]:

print(predict_customer(input_data))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step
(np.float32(0.028697744), 'Customer will stay')
